# Duplicate audit (CPU): what does each rule call a near-duplicate?

Input: **APTOS 2019 Blindness Detection** competition. Internet ON. Crops all 3,662 images to 256 px,
then compares the pHash rule with a vessel-pattern correlation and shows contact sheets of flagged pairs.

In [ ]:
REPO = "https://github.com/nabirakhan/genai"

In [ ]:
import os, sys, glob, re, json, time, shutil, signal, subprocess
WORK = "/kaggle/working"; CODE = f"{WORK}/genai"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--recurse-submodules"], check=False)
os.chdir(CODE)
subprocess.run([sys.executable, "scripts/apply_patches.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "click", "requests", "tqdm", "imagehash", "opencv-python-headless", "scikit-learn", "pandas", "scipy", "pillow"], check=True)
for opt in ():    # speed-ups only; stylegan3 falls back without them
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", opt]).returncode:
        print("optional package not installed:", opt)
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch; print("torch", torch.__version__, "gpus", torch.cuda.device_count())
def sh(cmd, **kw):
    print("$", " ".join(cmd)); return subprocess.run(cmd, check=True, **kw)


In [ ]:
DATA = f"{WORK}/data"; OUT = f"{WORK}/out"
os.environ["GENAI_DATA"] = DATA
os.makedirs(f"{DATA}/raw/aptos", exist_ok=True)
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), "train_images"))]
assert found, "add the APTOS 2019 competition as input"
if not os.path.exists(f"{DATA}/raw/aptos/train_images"):
    os.symlink(f"{found[0]}/train_images", f"{DATA}/raw/aptos/train_images")
shutil.copy(f"{found[0]}/train.csv", f"{DATA}/raw/aptos/train.csv")
sh([sys.executable, "-m", "src.data.prepare_images", "--source", "aptos"])
sh([sys.executable, "-m", "src.data.dedup_audit", "--out", f"{OUT}/audit"])

In [ ]:
from IPython.display import Image as IImage, display
print(open(f"{OUT}/audit/summary.txt").read())
display(IImage(f"{OUT}/audit/sheet_phash_pairs.png"))
display(IImage(f"{OUT}/audit/sheet_other_candidates.png"))
os.chdir(WORK)
shutil.rmtree(DATA, ignore_errors=True)
shutil.rmtree(CODE, ignore_errors=True)